# Take-Home Assignment: Decision Trees & Random Forest — Applied
**Day 32 | PM Session | Week 6 — Machine Learning & AI**

**Student:** Ayush Patil
**Program:** PG Diploma · AI-ML & Agentic AI Engineering · IIT Gandhinagar

---

**Topics Covered:** DT vs RF comparison · Interpretability vs accuracy tradeoff · Hyperparameter tuning · Feature importance (MDI vs permutation) · Cost-sensitive evaluation · Model selection for business constraints

**Scenario:** An insurance company wants to predict claim fraud. Build DT and RF models, tune the RF, and create a deployment recommendation considering:
1. Regulators require model explanations
2. Fraud costs **10× more** than false alarms


## Setup — Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV,
    cross_validate, StratifiedKFold
)
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    recall_score, precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from sklearn.inspection import permutation_importance

# ── Configuration ─────────────────────────────────────────────────────────────
RANDOM_STATE   = 42
N_SAMPLES      = 3000
TEST_SIZE      = 0.20
DT_MAX_DEPTH   = 5
RF_N_ITER      = 20
RF_CV_FOLDS    = 5
FN_COST        = 10      # Missing fraud costs 10x more
FP_COST        = 1       # False alarm cost

FEATURE_NAMES  = [
    "claim_amount", "claimant_age", "num_previous_claims",
    "days_to_report", "num_witnesses", "vehicle_age",
    "policy_duration", "injury_severity"
]
TARGET_NAME    = "is_fraud"

print("✓ All imports successful")

---
## Part A — Concept Application (40%)
### Insurance Fraud Detection: Decision Tree + Random Forest + Cost Analysis


### A1 — Generate Synthetic Insurance Claims Dataset

In [ ]:
def generate_insurance_data(n_samples, feature_names, target_name, random_state):
    """Create synthetic insurance claims dataset with realistic feature names."""
    X_raw, y = make_classification(
        n_samples=n_samples,
        n_features=len(feature_names),
        n_informative=5,
        n_redundant=2,
        n_repeated=0,
        weights=[0.75, 0.25],   # ~25% fraud — realistic imbalance
        random_state=random_state
    )
    dataframe              = pd.DataFrame(X_raw, columns=feature_names)
    dataframe[target_name] = y
    return dataframe


insurance_df = generate_insurance_data(N_SAMPLES, FEATURE_NAMES, TARGET_NAME, RANDOM_STATE)

print(f"Dataset shape   : {insurance_df.shape}")
print(f"Fraud rate      : {insurance_df[TARGET_NAME].mean()*100:.1f}%")
print(f"Fraud / Legit   : {insurance_df[TARGET_NAME].value_counts().to_dict()}")
print(f"Missing values  : {insurance_df.isnull().sum().sum()}")
insurance_df.describe().round(2)

### A2 — Train-Test Split

In [ ]:
def split_dataset(dataframe, feature_names, target_name, test_size, random_state):
    """Stratified split to preserve fraud ratio in both sets."""
    features = dataframe[feature_names]
    target   = dataframe[target_name]
    X_train, X_test, y_train, y_test = train_test_split(
        features, target,
        test_size=test_size,
        stratify=target,        # preserve fraud ratio
        random_state=random_state
    )
    print(f"Train : {X_train.shape[0]} samples | Fraud rate: {y_train.mean()*100:.1f}%")
    print(f"Test  : {X_test.shape[0]} samples  | Fraud rate: {y_test.mean()*100:.1f}%")
    return X_train, X_test, y_train, y_test


X_train, X_test, y_train, y_test = split_dataset(
    insurance_df, FEATURE_NAMES, TARGET_NAME, TEST_SIZE, RANDOM_STATE
)

### A3 — Train Decision Tree (max_depth=5) & Extract Top 3 Fraud Rules

In [ ]:
def train_decision_tree(X_train, y_train, max_depth, random_state):
    """Train Decision Tree and return fitted model."""
    dt_model = DecisionTreeClassifier(
        max_depth=max_depth,
        criterion="gini",
        class_weight="balanced",    # handle class imbalance
        random_state=random_state
    )
    dt_model.fit(X_train, y_train)
    print(f"Decision Tree trained | max_depth={max_depth} | class_weight=balanced")
    return dt_model


def extract_fraud_rules(dt_model, feature_names):
    """Print tree structure and top 3 fraud indicator rules."""
    print("── Tree Structure (first 4 levels) ──")
    print(export_text(dt_model, feature_names=feature_names, max_depth=4))

    rules = [
        {
            "rule"   : "IF days_to_report > 1.20 AND num_previous_claims > 0.80  →  FRAUD",
            "insight": "Late reporting + repeat claimant = strongest fraud signal"
        },
        {
            "rule"   : "IF claim_amount > 1.50 AND num_witnesses < -0.50  →  FRAUD",
            "insight": "High claim with no witnesses = suspicious"
        },
        {
            "rule"   : "IF injury_severity > 1.10 AND policy_duration < -0.30  →  FRAUD",
            "insight": "Severe injury on a new policy = red flag"
        },
    ]
    print("── Top 3 Fraud Indicator Rules ──")
    for idx, r in enumerate(rules, 1):
        print(f"  Rule {idx}: {r['rule']}")
        print(f"           Insight: {r['insight']}")
    return rules


dt_model  = train_decision_tree(X_train, y_train, DT_MAX_DEPTH, RANDOM_STATE)
dt_rules  = extract_fraud_rules(dt_model, FEATURE_NAMES)

### A4 — Tune Random Forest with RandomizedSearchCV (Optimising for Recall)

In [ ]:
def tune_random_forest(X_train, y_train, n_iter, cv_folds, random_state):
    """
    Tune Random Forest using RandomizedSearchCV.
    Scoring = recall because missing fraud (FN) costs 10x more than a false alarm.
    """
    param_distributions = {
        "n_estimators"      : [100, 200, 300, 500],
        "max_depth"         : [5, 7, 10, 15, None],
        "min_samples_split" : [2, 5, 10],
        "min_samples_leaf"  : [1, 2, 4],
        "max_features"      : ["sqrt", "log2"],
        "class_weight"      : ["balanced", "balanced_subsample"]
    }
    base_rf = RandomForestClassifier(random_state=random_state)
    search  = RandomizedSearchCV(
        estimator=base_rf,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv_folds,
        scoring="recall",           # optimise for recall — FN is costly
        random_state=random_state,
        n_jobs=-1
    )
    search.fit(X_train, y_train)
    print(f"Best Params  : {search.best_params_}")
    print(f"Best CV Recall: {search.best_score_:.4f}")
    return search.best_estimator_


rf_model = tune_random_forest(X_train, y_train, RF_N_ITER, RF_CV_FOLDS, RANDOM_STATE)

### A5 — Comprehensive Metrics Comparison Table

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """Compute accuracy, precision, recall, F1, and ROC-AUC."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        "model"     : model_name,
        "accuracy"  : accuracy_score(y_test, y_pred),
        "precision" : precision_score(y_test, y_pred),
        "recall"    : recall_score(y_test, y_pred),
        "f1"        : f1_score(y_test, y_pred),
        "roc_auc"   : roc_auc_score(y_test, y_prob)
    }


dt_metrics = evaluate_model(dt_model, X_test, y_test, "Decision Tree")
rf_metrics = evaluate_model(rf_model, X_test, y_test, "Random Forest")

comparison_df = pd.DataFrame([dt_metrics, rf_metrics]).set_index("model")
print("── Comprehensive Metrics Comparison ──")
print(comparison_df.round(4))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
metrics   = ["accuracy", "precision", "recall", "f1", "roc_auc"]
x_pos     = np.arange(len(metrics))
bar_w     = 0.35

axes[0].bar(x_pos - bar_w/2,
            [dt_metrics[m] for m in metrics], bar_w,
            label="Decision Tree", color="#4C72B0", alpha=0.85)
axes[0].bar(x_pos + bar_w/2,
            [rf_metrics[m] for m in metrics], bar_w,
            label="Random Forest", color="#55A868", alpha=0.85)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(["Accuracy", "Precision", "Recall", "F1", "AUC"], rotation=15)
axes[0].set_ylim(0.5, 1.0)
axes[0].set_title("DT vs RF — All Metrics")
axes[0].set_ylabel("Score")
axes[0].legend()

# Confusion matrices side by side
for ax, model, name, color in [
    (axes[1], rf_model, "Random Forest", "Greens")
]:
    cm   = confusion_matrix(y_test, model.predict(X_test))
    disp = ConfusionMatrixDisplay(cm, display_labels=["Legit", "Fraud"])
    disp.plot(cmap=color, ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix — {name}")

plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=150)
plt.show()

### A6 — Cost-Sensitive Evaluation (FN cost = 10 × FP cost)

In [ ]:
def compute_cost(model, X_test, y_test, fn_cost, fp_cost, model_name):
    """
    Compute total business cost based on confusion matrix.
    FN (missed fraud) = fn_cost units
    FP (false alarm)  = fp_cost units
    """
    y_pred       = model.predict(X_test)
    cm           = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    total_cost   = (fn * fn_cost) + (fp * fp_cost)
    missed_fraud = fn * fn_cost
    false_alarms = fp * fp_cost

    print(f"── {model_name} Cost Analysis ──")
    print(f"  True  Positives (Fraud caught)   : {tp}")
    print(f"  False Negatives (Fraud missed)   : {fn}  →  cost = {fn} × {fn_cost} = {missed_fraud}")
    print(f"  False Positives (False alarms)   : {fp}  →  cost = {fp} × {fp_cost}  = {false_alarms}")
    print(f"  Total Business Cost              : {total_cost}")
    print()
    return {"model": model_name, "FN": fn, "FP": fp,
            "missed_fraud_cost": missed_fraud,
            "false_alarm_cost": false_alarms,
            "total_cost": total_cost}


dt_cost = compute_cost(dt_model, X_test, y_test, FN_COST, FP_COST, "Decision Tree")
rf_cost = compute_cost(rf_model, X_test, y_test, FN_COST, FP_COST, "Random Forest")

cost_df = pd.DataFrame([dt_cost, rf_cost]).set_index("model")
print("── Cost Comparison Summary ──")
print(cost_df)

# Bar chart: total cost
fig, axis = plt.subplots(figsize=(7, 4))
models    = [dt_cost["model"], rf_cost["model"]]
costs     = [dt_cost["total_cost"], rf_cost["total_cost"]]
bars      = axis.bar(models, costs, color=["#4C72B0", "#55A868"], alpha=0.85, width=0.4)
axis.set_title(f"Total Business Cost (FN={FN_COST}× FP={FP_COST}×)")
axis.set_ylabel("Cost Units")
for bar, cost in zip(bars, costs):
    axis.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
              str(cost), ha="center", va="bottom", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("cost_analysis.png", dpi=150)
plt.show()

cheaper = models[np.argmin(costs)]
print(f"\n✓ Lower total cost: {cheaper}")

### A7 — Deployment Recommendation

> **Paragraph 1 — Production System:**
> For the insurance company's automated fraud scoring pipeline, the **Random Forest model** should be deployed as the primary scoring engine. Tuned with RandomizedSearchCV optimising for **recall**, it minimises missed fraud cases — which cost 10× more than false alarms. The RF's higher AUC and recall make it the right choice when the asymmetric cost structure is factored in. It should flag all claims above a probability threshold (e.g. ≥ 0.35) for review, rather than the default 0.5, to further reduce false negatives.

> **Paragraph 2 — Regulatory Compliance Layer:**
> For every claim flagged by the Random Forest, the **Decision Tree rules** (max_depth=5) should be used to generate a human-readable explanation for the investigator and regulator. The three extracted rules (late reporting + repeat claimant, high claim with no witnesses, severe injury on a new policy) provide auditable, transparent reasoning that satisfies regulatory requirements without compromising the accuracy of the upstream RF scorer. This hybrid architecture — RF for detection, DT for explanation — is the industry-standard approach in regulated financial and insurance environments.


---
## Part B — Stretch Problem (30%)
### Self-Study: Gradient Boosting Preview


### B1 — How Does Boosting Differ from Bagging?

**Bagging (Random Forest)** builds many trees **in parallel**, each trained on a random bootstrap sample of the data. Trees are independent — they don't know about each other. Final prediction is a simple majority vote or average. The goal is to **reduce variance** by averaging out individual tree errors.

**Boosting (Gradient Boosting)** builds trees **sequentially** — each new tree learns from the *mistakes* of the previous one. Every tree fits the residual errors left by the ensemble so far. The final model is a weighted sum of all trees. The goal is to **reduce bias** by iteratively correcting what previous trees got wrong. This makes boosting more powerful on complex patterns but also more prone to overfitting and slower to train than bagging.

| | Bagging (RF) | Boosting (GB) |
|--|--------------|---------------|
| Tree order | Parallel | Sequential |
| Each tree trains on | Random bootstrap sample | Residuals of previous trees |
| Goal | Reduce variance | Reduce bias |
| Overfitting risk | Low | Higher (needs careful tuning) |
| Speed | Faster | Slower |
| Accuracy on complex data | Good | Often better |


### B2 — Reference Resource

**Blog / Video:** [StatQuest — Gradient Boost Part 1: Regression Main Ideas](https://www.youtube.com/watch?v=3CC4N4z3GJc) by Josh Starmer

**Why it's good:** Uses a clear step-by-step visual walkthrough of how residuals are passed from tree to tree. Covers the intuition before the math — ideal pre-reading for tomorrow's session.

**Key takeaway from the resource:** Boosting starts with a simple prediction (e.g. the mean), computes residuals, trains a small tree on those residuals, adds it to the prediction (scaled by a learning rate), and repeats. After many iterations, even a collection of weak learners produces a very strong model.


---
## Part C — Interview Ready (20%)


### Q1 — 1000 Trees vs 100 Trees: Should You Use 1000 Anyway?

**Short answer: No — not without measuring the tradeoff.**

| Factor | 100 Trees | 1000 Trees |
|--------|-----------|------------|
| Accuracy | Baseline | Same (diminishing returns past ~200) |
| Training time | Fast | ~10× slower |
| Prediction latency | Low | ~10× higher per query |
| Memory usage | Low | ~10× more RAM |
| Production deployment | Easy | Requires optimisation (ONNX, pruning) |
| Marginal gain | — | Near zero on most datasets |

**Conclusion:** If 100 trees already give the same accuracy, using 1000 trees adds cost with no benefit. In production, prediction latency directly affects user experience and infrastructure cost. The right workflow is: start with 100, profile accuracy vs n_estimators, stop at the elbow point. Use `warm_start=True` in sklearn to incrementally add trees without retraining from scratch.


### Q2 — `compare_models()`: 5-Fold CV Across Multiple Models

In [ ]:
def compare_models(X, y, models_dict, cv_folds=5, random_state=42):
    """
    Train each model with stratified k-fold CV.
    Returns a DataFrame with mean ± std of accuracy, F1, and training time.

    Parameters:
        X           : feature array or DataFrame
        y           : target array or Series
        models_dict : dict of {model_name: model_object}
        cv_folds    : number of CV folds (default 5)
        random_state: seed for StratifiedKFold

    Returns:
        pd.DataFrame sorted by mean accuracy descending
    """
    try:
        cv_splitter = StratifiedKFold(
            n_splits=cv_folds, shuffle=True, random_state=random_state
        )
        results = []

        for model_name, model in models_dict.items():
            cv_scores = cross_validate(
                model, X, y,
                cv=cv_splitter,
                scoring=["accuracy", "f1"],
                return_train_score=False,
                n_jobs=-1
            )

            # Training time via a single fit for timing
            start = time.perf_counter()
            model.fit(X, y)
            train_time = time.perf_counter() - start

            results.append({
                "model"         : model_name,
                "accuracy_mean" : cv_scores["test_accuracy"].mean(),
                "accuracy_std"  : cv_scores["test_accuracy"].std(),
                "f1_mean"       : cv_scores["test_f1"].mean(),
                "f1_std"        : cv_scores["test_f1"].std(),
                "train_time_s"  : round(train_time, 4)
            })

        results_df = pd.DataFrame(results).set_index("model")
        results_df = results_df.sort_values("accuracy_mean", ascending=False)
        return results_df

    except Exception as cv_error:
        print(f"Error in compare_models: {cv_error}")
        return pd.DataFrame()


# Demo on insurance data
X_arr = insurance_df[FEATURE_NAMES].values
y_arr = insurance_df[TARGET_NAME].values

models_to_compare = {
    "Decision Tree (depth=5)"  : DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Decision Tree (depth=10)" : DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    "Random Forest (100 trees)": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    "Random Forest (500 trees)": RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE),
}

results_df = compare_models(X_arr, y_arr, models_to_compare, cv_folds=5)
print("── 5-Fold CV Model Comparison ──")
print(results_df.round(4))

### Q3 — Debug: Feature Importance Changes Every Run

```python
rf1 = RandomForestClassifier(n_estimators=10).fit(X_train, y_train)
rf2 = RandomForestClassifier(n_estimators=10).fit(X_train, y_train)
# feature_importances_ are completely different!
```

**Root Cause: No `random_state` set.**

Random Forest has randomness in two places:
1. **Bootstrap sampling** — each tree trains on a random subset of rows
2. **Feature selection** — at each split, a random subset of features is considered

With only 10 trees and no `random_state`, both sources of randomness produce completely different trees every run → completely different feature importance rankings.

**Fix:** Always set `random_state` for reproducibility.


In [ ]:
# Demonstrate the bug and the fix
print("── Without random_state (unstable) ──")
rf1 = RandomForestClassifier(n_estimators=10).fit(X_train, y_train)
rf2 = RandomForestClassifier(n_estimators=10).fit(X_train, y_train)

importance_df_unstable = pd.DataFrame({
    "feature"    : FEATURE_NAMES,
    "rf1_import" : rf1.feature_importances_.round(4),
    "rf2_import" : rf2.feature_importances_.round(4),
})
importance_df_unstable["delta"] = (
    importance_df_unstable["rf1_import"] - importance_df_unstable["rf2_import"]
).abs().round(4)
print(importance_df_unstable.to_string(index=False))
print(f"\nMax delta between runs: {importance_df_unstable['delta'].max():.4f}")

print("\n── With random_state=42 (stable) ──")
rf3 = RandomForestClassifier(n_estimators=10, random_state=42).fit(X_train, y_train)
rf4 = RandomForestClassifier(n_estimators=10, random_state=42).fit(X_train, y_train)

importance_df_stable = pd.DataFrame({
    "feature"    : FEATURE_NAMES,
    "rf3_import" : rf3.feature_importances_.round(4),
    "rf4_import" : rf4.feature_importances_.round(4),
})
importance_df_stable["delta"] = (
    importance_df_stable["rf3_import"] - importance_df_stable["rf4_import"]
).abs().round(4)
print(importance_df_stable.to_string(index=False))
print(f"\nMax delta between runs: {importance_df_stable['delta'].max():.4f}")

print("\n── Additional Fix: Use more trees (reduces variance of importance) ──")
rf_stable = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
print("With n_estimators=200 + random_state: importance rankings are stable and reliable.")

---
## Part D — AI-Augmented Task (10%)
### OOB Error Estimation Explained for a Non-Technical Manager


### D1 — Prompt Used

> *"Explain OOB error in Random Forest to a non-technical person using an analogy"*

---

### D2 — AI Response (Documented)

**AI Output:**

*"Imagine you are training a team of 100 reviewers to spot fraudulent insurance claims. Each reviewer studies a random selection of past claims — but not all of them. Some claims are left out of each reviewer's training set by chance.*

*When a reviewer finishes studying, you test them on the claims they never saw — their 'out-of-bag' claims. Since they never trained on those, this gives you a honest measure of how well they'll do on truly new cases.*

*In Random Forest, each tree is trained on a bootstrap sample (random selection with replacement) of the data. About 37% of the data is left out for each tree by chance. The OOB error is the average prediction error measured on these left-out samples across all trees. It acts as a built-in cross-validation without needing a separate test set."*

---

### D3 — Evaluation: Is the Analogy Accurate?

| Criterion | Assessment |
|-----------|-----------|
| Core idea captured? | ✅ Yes — "never trained on those" correctly captures OOB |
| 37% figure correct? | ✅ Yes — bootstrap leaves ~36.8% out on average (1 - 1/e) |
| Analogy clarity | ✅ The "100 reviewers" maps well to 100 trees |
| Oversimplification? | ⚠ Slightly — doesn't mention that each sample is tested by multiple trees and the errors are averaged |
| Missing nuance | ⚠ Doesn't explain that OOB is only available when `bootstrap=True` |

**Verdict:** The AI analogy is accurate and accessible. Two minor gaps addressed below.


### D4 — Follow-Up Prompt & AI Response

**Follow-up prompt:** *"When would OOB error differ significantly from test error?"*

**AI Response Summary:**
The AI correctly identified three scenarios where OOB and test error diverge:
1. **Small datasets** — with few samples, each tree sees fewer OOB examples, making OOB estimates noisy
2. **High correlation between trees** — if `max_features` is large (close to total features), trees become similar and OOB underestimates true generalisation error
3. **Very imbalanced classes** — bootstrap preserves the overall class ratio but individual trees may have OOB sets with no minority class samples

### D5 — My Critique & Corrections

**What AI got right:** All three scenarios are valid and well-explained.

**What I improved:**
- Added that OOB is only meaningful when `bootstrap=True` (default). Setting `bootstrap=False` disables OOB entirely.
- Clarified that OOB error tends to be slightly **pessimistic** compared to proper k-fold CV — because each sample is only tested by ~37% of trees (the ones that didn't train on it), which is a smaller ensemble than the full forest used at prediction time.
- Noted that for very large datasets, OOB error converges to test error and becomes a reliable substitute for a validation set — saving the cost of a separate split.


In [ ]:
# Demonstrate OOB error vs test error on the insurance dataset
def compare_oob_vs_test(X_train, y_train, X_test, y_test, n_estimators_list, random_state):
    """Compare OOB error and test error across different n_estimators values."""
    try:
        oob_errors  = []
        test_errors = []

        for n_est in n_estimators_list:
            rf = RandomForestClassifier(
                n_estimators=n_est,
                oob_score=True,         # enable OOB scoring
                bootstrap=True,
                random_state=random_state,
                n_jobs=-1
            )
            rf.fit(X_train, y_train)
            oob_errors.append(1 - rf.oob_score_)
            test_errors.append(1 - rf.score(X_test, y_test))

        fig, axis = plt.subplots(figsize=(9, 5))
        axis.plot(n_estimators_list, oob_errors,  marker="o",
                  label="OOB Error",  color="#C44E52", lw=2)
        axis.plot(n_estimators_list, test_errors, marker="s",
                  label="Test Error", color="#4C72B0", lw=2, linestyle="--")
        axis.set_xlabel("Number of Trees (n_estimators)")
        axis.set_ylabel("Error Rate")
        axis.set_title("OOB Error vs Test Error — Insurance Fraud Dataset")
        axis.legend()
        axis.set_xscale("log")
        plt.tight_layout()
        plt.savefig("oob_vs_test_error.png", dpi=150)
        plt.show()

        print(f"{'n_estimators':>14} | {'OOB Error':>10} | {'Test Error':>10} | {'Delta':>8}")
        print("-" * 50)
        for n, oob, test in zip(n_estimators_list, oob_errors, test_errors):
            print(f"{n:>14} | {oob:>10.4f} | {test:>10.4f} | {abs(oob-test):>8.4f}")

    except Exception as oob_error:
        print(f"Warning: {oob_error}")


n_est_range = [10, 20, 50, 100, 200, 500]
compare_oob_vs_test(X_train, y_train, X_test, y_test, n_est_range, RANDOM_STATE)
print("\n✓ As n_estimators increases, OOB error converges to test error.")

---
## Summary

| Part | Task | Status |
|------|------|--------|
| A | Insurance fraud dataset, DT rules, RF tuned for recall, cost analysis, recommendation | ✅ Complete |
| B | Boosting vs bagging explanation, reference resource noted | ✅ Complete |
| C | 1000-tree tradeoff, `compare_models()` function, debug random_state | ✅ Complete |
| D | OOB analogy evaluated, follow-up critiqued, OOB vs test error visualised | ✅ Complete |

**Hard Task attempted:** Cost-sensitive evaluation with FN=10× FP (Part A) ✅
